---
# Clinical Data Quality Engine (DQIE)
# Notebook 01 — Ingestion Testing
# Purpose: Validate ingestion functions for CSV, SQL, OCR, JSON, and images
---


# Preparations
---

## Setup do ambiente

In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd

# Add project root to PYTHONPATH
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print("PYTHONPATH OK:", PROJECT_ROOT)


## Imports of ingestion modules

In [ ]:
from src._2_ingestion.load_csv import CSVIngestion
from src._2_ingestion.load_sql import SQLIngestion
from src._2_ingestion.load_ocr import OCRIngestion
from src._2_ingestion.load_ocr_images import OCRImageIngestion

## Loaders

In [ ]:
csv_loader = CSVIngestion(base_path="../data/_1_bronze/csv")
sql_loader = SQLIngestion(sql_base_path="../data/_1_bronze/sql")
ocr_json_loader = OCRIngestion(base_path="../data/_1_bronze/ocr")
ocr_image_loader = OCRImageIngestion(images_path="../data/_1_bronze/images/ocr_reports")



# Load Data
---

## Load CSV

In [ ]:
df_patients = csv_loader.load("patients.csv")
df_injuries = csv_loader.load("injuries.csv")
df_sessions = csv_loader.load("sessions.csv")

df_patients.head()


## Load SQL

In [ ]:
df_sql_patients = sql_loader.load("patients.sql")
df_sql_sessions = sql_loader.load("sessions.sql")

df_sql_patients.head()


## Load OCR JSON

In [ ]:
df_clinical_reports = ocr_json_loader.load("clinical_reports.json")
df_ocr_extracted = ocr_json_loader.load("ocr_extracted.json")

df_clinical_reports.head()


## Load OCR Images

In [ ]:
df_ocr_images = ocr_image_loader.load()
df_ocr_images.head()


# Basic Checks

## Basic sanity checks

In [ ]:
print("Patients CSV:", df_patients.shape)
print("Injuries CSV:", df_injuries.shape)
print("Sessions CSV:", df_sessions.shape)

print("Patients SQL:", df_sql_patients.shape)
print("Sessions SQL:", df_sql_sessions.shape)

print("Clinical Reports JSON:", df_clinical_reports.shape)
print("OCR Extracted JSON:", df_ocr_extracted.shape)

print("OCR Images:", df_ocr_images.shape)

## Check column consistency

In [ ]:
def check_column_consistency(df1, df2, name1, name2):
    cols1 = set(df1.columns)
    cols2 = set(df2.columns)

    print(f"\n🔍 Column consistency: {name1} vs {name2}")
    print("Only in", name1, ":", cols1 - cols2)
    print("Only in", name2, ":", cols2 - cols1)
    print("Intersection:", cols1 & cols2)

### Checks CSV vs SQL

In [ ]:
check_column_consistency(df_patients, df_sql_patients, "patients.csv", "patients.sql")
check_column_consistency(df_sessions, df_sql_sessions, "sessions.csv", "sessions.sql")


### Checks OCR JSON vs OCR Images

In [ ]:
check_column_consistency(df_ocr_extracted, df_ocr_images, "ocr_extracted.json", "ocr_images")


## Quick data profiling

In [ ]:
def quick_profile(df, name):
    print(f"\n📊 Quick Profile — {name}")
    print("Shape:", df.shape)
    print("Columns:", list(df.columns))
    print("\nMissing values:")
    print(df.isna().sum())
    print("\nSample rows:")
    display(df.head(3))


In [ ]:
quick_profile(df_patients, "patients.csv")
quick_profile(df_injuries, "injuries.csv")
quick_profile(df_sessions, "sessions.csv")

quick_profile(df_sql_patients, "patients.sql")
quick_profile(df_sql_sessions, "sessions.sql")

quick_profile(df_clinical_reports, "clinical_reports.json")
quick_profile(df_ocr_extracted, "ocr_extracted.json")

quick_profile(df_ocr_images, "ocr_images (EasyOCR)")


# Saving

## Save ingestion outputs for next notebooks

In [ ]:
SILVER_DIR = Path("../data/_2_silver")
SILVER_DIR.mkdir(exist_ok=True)
print("Silver directory:", SILVER_DIR)


In [ ]:
def save_silver(df, name):
    path = SILVER_DIR / f"{name}.parquet"
    df.to_parquet(path, index=False)
    print(f"Saved Silver: {path}")


In [ ]:
save_silver(df_patients, "patients")
save_silver(df_injuries, "injuries")
save_silver(df_sessions, "sessions")

save_silver(df_sql_patients, "patients_sql")
save_silver(df_sql_sessions, "sessions_sql")

save_silver(df_clinical_reports, "clinical_reports")
save_silver(df_ocr_extracted, "ocr_extracted")

save_silver(df_ocr_images, "ocr_images")
